# 09. La FFT en 2D: imágenes en el dominio de la frecuencia

**Objetivos de aprendizaje.** Al terminar podrás:

1. Explicar qué es una **frecuencia espacial** (ciclos/pixel) y su dirección.
2. Calcular la FFT 2D como **FFT de las filas y luego de las columnas**.
3. Leer un espectro 2D centrado (`fftshift`): centro = frecuencias bajas, orillas = frecuencias altas.
4. Filtrar imágenes en frecuencia: **pasa-bajas** (suavizar) y **pasa-altas** (bordes).
5. Relacionar la **convolución 2D** con la multiplicación en frecuencia.

**Requisito:** notebooks 03 y 08.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## 1. Frecuencia espacial

En una señal de audio, la frecuencia dice qué tan rápido cambia la señal **en el tiempo**. En una imagen,
dice qué tan rápido cambia la intensidad **en el espacio**: se mide en **ciclos por pixel**, y además tiene **dirección**.

$$x[m, n] = \cos\big(2\pi (f_m\, m + f_n\, n)\big)$$

- $f_n$: ciclos por pixel en la dirección **horizontal** (a lo largo de las columnas $n$).
- $f_m$: ciclos por pixel en la dirección **vertical** (a lo largo de las filas $m$).

In [ ]:
def imagen_coseno(tamano, ciclos_horizontales, ciclos_verticales):
    '''Imagen tamano x tamano con un coseno 2D que completa el número de ciclos indicado.'''
    m = np.arange(tamano)                    # índices de fila
    n = np.arange(tamano)                    # índices de columna
    imagen = np.zeros((tamano, tamano))
    for i in range(tamano):                  # para cada fila
        for j in range(tamano):              # para cada columna
            fase = 2 * np.pi * (ciclos_verticales * m[i] + ciclos_horizontales * n[j]) / tamano
            imagen[i, j] = np.cos(fase)
    return imagen


figura, ejes = plt.subplots(1, 4, figsize=(15, 4))
casos = [(4, 0), (16, 0), (0, 8), (8, 8)]
for eje, (ch, cv) in zip(ejes, casos):
    pds.mostrar_imagen(imagen_coseno(128, ch, cv), titulo=f"{ch} ciclos horiz., {cv} vert.", ax=eje)
plt.tight_layout()
plt.show()

## 2. La FFT 2D paso a paso

$$X[k, l] = \sum_{m=0}^{M-1}\sum_{n=0}^{N-1} x[m,n]\, e^{-j2\pi\left(\frac{km}{M} + \frac{ln}{N}\right)}$$

Se puede separar en dos pasos con FFT de **una** dimensión:

1. FFT de **cada fila** (frecuencias horizontales).
2. FFT de **cada columna** del resultado (frecuencias verticales).

In [ ]:
imagen = imagen_coseno(64, 5, 0) + 0.5 * imagen_coseno(64, 0, 3)
filas, columnas = imagen.shape

# Paso 1: FFT de cada fila
paso1 = np.zeros((filas, columnas), dtype=complex)
for i in range(filas):
    paso1[i, :] = np.fft.fft(imagen[i, :])

# Paso 2: FFT de cada columna
paso2 = np.zeros((filas, columnas), dtype=complex)
for j in range(columnas):
    paso2[:, j] = np.fft.fft(paso1[:, j])

pds.comprobar_iguales(paso2, np.fft.fft2(imagen), "filas y luego columnas = np.fft.fft2")

## 3. Centrar el espectro: `fftshift`

Igual que en 1D, la segunda mitad de los índices corresponde a frecuencias **negativas**. `np.fft.fftshift` reacomoda
el espectro para que la frecuencia $(0, 0)$ quede en el **centro** de la imagen:

- **Centro** → frecuencias bajas (zonas uniformes, el brillo promedio).
- **Lejos del centro** → frecuencias altas (bordes, detalles finos, ruido).
- Un coseno 2D aparece como **dos puntos simétricos** respecto al centro, en la **dirección** en que oscila.

Usamos $\log(1 + |X|)$ para poder ver valores muy distintos en la misma imagen.

In [ ]:
import inspect
print(inspect.getsource(pds.espectro_2d))

In [ ]:
figura, ejes = plt.subplots(2, 4, figsize=(15, 7))
for c, (ch, cv) in enumerate(casos):
    img = imagen_coseno(128, ch, cv)
    _, espectro_log = pds.espectro_2d(img)
    pds.mostrar_imagen(img, titulo=f"{ch} horiz., {cv} vert.", ax=ejes[0, c])
    pds.mostrar_imagen(espectro_log, titulo="espectro (centrado)", ax=ejes[1, c])
    ejes[1, c].set_xlim(64 - 24, 64 + 24)
    ejes[1, c].set_ylim(64 + 24, 64 - 24)
plt.tight_layout()
plt.show()

> Nota: Observa: los puntos se alejan del centro cuando hay **más** ciclos (frecuencia más alta), y la línea que los une
> es **perpendicular a las franjas** (es la dirección en que la imagen cambia).

## 4. Espectros de imágenes reales

In [ ]:
prueba = pds.cargar_imagen("../datos/imagen_prueba.png")
foto = pds.cargar_imagen("../datos/grace_hopper.jpg")

figura, ejes = plt.subplots(2, 2, figsize=(11, 10))
for fila, img in enumerate([prueba, foto]):
    _, espectro_log = pds.espectro_2d(img)
    pds.mostrar_imagen(img, titulo="imagen", ax=ejes[fila, 0])
    pds.mostrar_imagen(espectro_log, titulo="log(1 + |FFT2|)", ax=ejes[fila, 1])
plt.tight_layout()
plt.show()

- En la imagen de prueba: las **franjas** (periodo 8 → $f = 1/8$) dan puntos brillantes a $256/8 = 32$ pixeles del centro
  en dirección horizontal. Los **bordes rectos** del rectángulo dan líneas brillantes perpendiculares a ellos.
- En la foto: casi toda la energía está **cerca del centro**. Las imágenes naturales son mayormente "suaves".

## 5. Filtrado en frecuencia

**Pasa-bajas:** conservamos sólo un círculo central → quitamos detalles → imagen **borrosa**.
**Pasa-altas:** quitamos el círculo central → conservamos los cambios rápidos → **bordes**.

Construyamos el filtro pasa-bajas en 4 pasos: FFT centrada → máscara circular → multiplicar → regresar al espacio.

In [ ]:
def filtrar_imagen_pasabajas(imagen, radio):
    '''
    Filtro pasa-bajas IDEAL: conserva sólo las frecuencias dentro de un
    círculo de radio 'radio' alrededor del centro del espectro.
    Regresa la imagen filtrada y la máscara.
    '''
    filas = imagen.shape[0]
    columnas = imagen.shape[1]

    # Paso 1: FFT 2D centrada
    F_centrada, _ = pds.espectro_2d(imagen)

    # Paso 2: máscara = 1 dentro del círculo, 0 fuera
    centro_fila = filas // 2
    centro_columna = columnas // 2
    mascara = np.zeros((filas, columnas))
    for i in range(filas):
        for j in range(columnas):
            distancia = np.sqrt((i - centro_fila) ** 2 + (j - centro_columna) ** 2)
            if distancia <= radio:
                mascara[i, j] = 1.0

    # Paso 3: multiplicar espectro por máscara
    F_filtrada = F_centrada * mascara

    # Paso 4: deshacer el shift y regresar al espacio con la IFFT 2D
    F_sin_shift = np.fft.ifftshift(F_filtrada)
    imagen_filtrada = np.real(np.fft.ifft2(F_sin_shift))

    return imagen_filtrada, mascara

In [ ]:
suave, mascara = filtrar_imagen_pasabajas(foto, radio=30)

# Pasa-altas: la máscara complementaria (1 - máscara)
F_centrada, _ = pds.espectro_2d(foto)
F_altas = F_centrada * (1 - mascara)
bordes = np.real(np.fft.ifft2(np.fft.ifftshift(F_altas)))

figura, ejes = plt.subplots(1, 4, figsize=(16, 4.5))
pds.mostrar_imagen(foto, titulo="original", ax=ejes[0])
pds.mostrar_imagen(mascara, titulo="máscara pasa-bajas (r = 30)", ax=ejes[1])
pds.mostrar_imagen(suave, titulo="pasa-bajas: borrosa", ax=ejes[2])
pds.mostrar_imagen(np.abs(bordes), titulo="pasa-altas: bordes", ax=ejes[3])
plt.tight_layout()
plt.show()

> Ojo: el filtro **ideal** (corte abrupto) produce "ondulaciones" alrededor de los bordes (*ringing*), por el mismo motivo
> que la fuga espectral en 1D. Los filtros prácticos usan transiciones suaves (por ejemplo, gaussianas).

## 6. Un adelanto: filtrar en el espacio con convolución 2D

Todo lo que hicimos en frecuencia (multiplicar el espectro por una máscara) también se puede hacer en el espacio,
con una **convolución 2D** entre la imagen y una matriz pequeña llamada *kernel*. Un kernel de promedio actúa como
pasa-bajas y uno de Sobel como pasa-altas. Este tema se desarrolla completo en el notebook 10.

In [ ]:
desenfoque = pds.convolucion_2d(foto, pds.kernel_promedio(9))
bordes_v = pds.convolucion_2d(foto, pds.kernel_sobel("horizontal"))

figura, ejes = plt.subplots(1, 3, figsize=(15, 4.5))
pds.mostrar_imagen(foto, titulo="original", ax=ejes[0])
pds.mostrar_imagen(desenfoque, titulo="promedio 9x9 (pasa-bajas)", ax=ejes[1])
pds.mostrar_imagen(np.abs(bordes_v), titulo="Sobel: bordes verticales (pasa-altas)", ax=ejes[2])
plt.tight_layout()
plt.show()

## Resumen

| 1D (audio, ECG) | 2D (imágenes) |
|---|---|
| $x[n]$ | $x[m, n]$ |
| frecuencia en ciclos/muestra | frecuencia en ciclos/pixel, con **dirección** |
| `np.fft.fft` | `np.fft.fft2` = FFT de filas y luego columnas |
| espejo en $f_s/2$ | `fftshift` → $(0,0)$ al centro |
| convolución $x*h$ | convolución 2D con un kernel |
| $Y = X\cdot H$ | $Y[k,l] = X[k,l]\cdot H[k,l]$ |

## Comprueba tu comprensión

1. Una imagen tiene franjas **horizontales** (cambia de arriba a abajo). ¿Dónde aparecen los puntos del espectro?
2. ¿Qué parte del espectro conserva un filtro que "suaviza"?
3. ¿Qué representa el pixel central del espectro (sin shift: $X[0,0]$)?

<details><summary>Ver respuestas</summary>

1. Arriba y abajo del centro (sobre el eje vertical).
2. El centro (frecuencias bajas).
3. La suma de todos los pixeles: $N\cdot M$ veces el brillo promedio.
</details>